# A09 · The Transformer & GPT from Scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adib2024/aws-architect-path/blob/main/notebooks/a09_gpt_from_scratch.ipynb)

Put it all together: embeddings, pre-LN blocks of attention and MLP with residuals, a final LayerNorm and a tied LM head. You'll count GPT-2's 124M parameters from a formula, derive the 6N FLOPs-per-token rule, then write and train a GPT that generates text.

Part of the **AI Engineer Path** (https://adib2024.github.io/aws-architect-path/#a09). In Colab, use **Runtime → Change runtime type → T4 GPU** for anything that trains.

**You will:**
- Derive the per-block parameter count $12d^2 + 13d$ and reproduce GPT-2 small's 124,439,808.
- Explain training FLOPs $\approx 6N$ per token and inference $\approx 2N$.
- Write a complete GPT in about 80 lines of PyTorch, train it and sample from it.
- Use temperature and top-k when sampling.

## The math

### Parameters per block

$$\underbrace{3d^2+3d}_{W_{QKV}} + \underbrace{d^2+d}_{W_O} + \underbrace{4d^2+4d}_{\text{MLP up}} + \underbrace{4d^2+d}_{\text{MLP down}} + \underbrace{4d}_{2\times\text{LN}} = 12d^2 + 13d$$

Attention holds $4d^2$ and the MLP $8d^2$. Total $N = Vd + T_{ctx}d + L(12d^2 + 13d) + 2d$ (final LN, tied head).

**Worked example**

1. GPT-2 small: $d = 768,\ L = 12,\ V = 50{,}257,\ T_{ctx} = 1024$
2. per block: $12\cdot768^2 + 13\cdot768 = 7{,}087{,}872$
3. blocks: $12 \times 7{,}087{,}872 = 85{,}054{,}464$
4. embeddings: $50{,}257\cdot768 + 1024\cdot768 = 39{,}383{,}808$
5. total $= 85{,}054{,}464 + 39{,}383{,}808 + 1{,}536 = 124{,}439{,}808$ (exactly what PyTorch reports)

### Training compute: the 6N rule

$$C_{\text{train}} \approx 6\,N\,D, \qquad C_{\text{infer}} \approx 2\,N \text{ FLOPs per token}$$

$N$ = non-embedding parameters, $D$ = training tokens. Ignores the attention $T^2$ term, which is small when $d \gg T/6$.

**Worked example**

1. GPT-2 small on 10B tokens: $6 \cdot 1.24\times10^8 \cdot 10^{10} = 7.5\times10^{18}$ FLOPs
2. one A10G at ~35 TFLOP/s bf16 achieved: $7.5\times10^{18}/3.5\times10^{13} \approx 2.1\times10^5$ s (≈ 2.5 days)

### Sanity check: the initial loss

$$L_0 \approx -\log\frac{1}{V} = \log V$$

A correctly initialised model predicts roughly uniformly at step 0.

**Worked example**

1. character-level Shakespeare, $V = 65$: $\ln 65 = 4.17$ (if you see 6 or 10, something is wrong)
2. GPT-2 vocabulary: $\ln 50{,}257 = 10.8$

## Lab: Write and train your own GPT

A character-level GPT on Shakespeare (swap in any Malay text file to train a Malay model). Uses the attention module from A08.

### Step 1

Data: download text, build a character vocabulary and a batch sampler.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math, urllib.request
device = 'cuda' if torch.cuda.is_available() else 'cpu'
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(url).read().decode()
chars = sorted(set(text)); vocab = len(chars)
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text])
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]
ctx, B = 128, 64
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - ctx - 1, (B,))
    x = torch.stack([d[i:i + ctx] for i in ix])
    y = torch.stack([d[i + 1:i + ctx + 1] for i in ix])      # targets = inputs shifted by one
    return x.to(device), y.to(device)
print(len(text), "characters,", vocab, "unique")

### Step 2

The model: attention (from A08), MLP, block, GPT with tied weights and generation.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h = h
        self.qkv, self.proj = nn.Linear(d, 3 * d), nn.Linear(d, d)
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = (t.view(B, T, self.h, C // self.h).transpose(1, 2) for t in self.qkv(x).split(C, dim=2))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).contiguous().view(B, T, C))

class MLP(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc, self.proj = nn.Linear(d, 4 * d), nn.Linear(4 * d, d)
    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))

class Block(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.ln1, self.attn = nn.LayerNorm(d), CausalSelfAttention(d, h)
        self.ln2, self.mlp = nn.LayerNorm(d), MLP(d)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab, ctx, d=384, h=6, L=6):
        super().__init__()
        self.ctx = ctx
        self.tok, self.pos = nn.Embedding(vocab, d), nn.Embedding(ctx, d)
        self.blocks = nn.ModuleList([Block(d, h) for _ in range(L)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab, bias=False)
        self.head.weight = self.tok.weight                    # weight tying
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for blk in self.blocks:
            x = blk(x)
        logits = self.head(self.ln_f(x))
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, n_new, temperature=1.0, top_k=None):
        for _ in range(n_new):
            logits, _ = self(idx[:, -self.ctx:])
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')
            nxt = torch.multinomial(logits.softmax(-1), num_samples=1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx

model = GPT(vocab, ctx).to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")   # 10,721,664 for vocab=65

### Step 3

Check the formula: per block $12d^2 + 13d$, and the initial loss $\approx \ln V$.

In [ ]:
d = 384
blk = sum(p.numel() for p in model.blocks[0].parameters())
print(blk, 12 * d * d + 13 * d)                      # 1774464 1774464
x, y = get_batch('train')
print(model(x, y)[1].item(), math.log(vocab))       # ~4.2 vs 4.17

### Step 4

Train with AdamW for 2,000 steps, evaluating validation loss along the way.

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, betas=(0.9, 0.95), weight_decay=0.1)
@torch.no_grad()
def eval_loss(split, iters=20):
    model.eval()
    out = torch.stack([model(*get_batch(split))[1] for _ in range(iters)]).mean().item()
    model.train(); return out
for step in range(2001):
    x, y = get_batch('train')
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 250 == 0:
        print(f"step {step:4d}  train {loss.item():.3f}  val {eval_loss('val'):.3f}")
# expect val loss to fall from ~4.2 to roughly 1.6-1.8

### Step 5

Generate. Try temperature 0.5 vs 1.2 and top_k=10.

In [ ]:
start = torch.tensor([[stoi['\n']]], device=device)
for temp in [0.5, 1.0, 1.2]:
    out = model.generate(start, 300, temperature=temp, top_k=None)
    print(f"--- temperature {temp} ---")
    print(''.join(itos[i] for i in out[0].tolist()))
torch.save(model.state_dict(), "gpt_char.pt")

### Clean up

Download `gpt_char.pt` if you want to keep it (you'll quantise and serve a model in A12), then disconnect the Colab runtime.

## Check yourself

**1. A model has $d = 2048,\ L = 24$. Approximate non-embedding parameters?**

A. 100M  
B. 1.2B  
C. 300M  
D. 12B

<details><summary>Answer</summary>

**B.** $12\cdot24\cdot2048^2 = 1.21\times10^9$.

</details>

**2. Training a 1B-parameter model on 20B tokens needs roughly…**

A. $1.2\times10^{20}$ FLOPs  
B. $4\times10^{19}$ FLOPs  
C. $2\times10^{10}$ FLOPs  
D. $1.2\times10^{23}$ FLOPs

<details><summary>Answer</summary>

**A.** $6ND = 6\cdot10^9\cdot2\times10^{10} = 1.2\times10^{20}$.

</details>

**3. Lower temperature (e.g. 0.3) when sampling makes output…**

A. More random  
B. More deterministic, sharper distribution  
C. Longer  
D. Use more memory

<details><summary>Answer</summary>

**B.** Dividing logits by $T<1$ magnifies differences before softmax, concentrating probability on top tokens.

</details>